In [1]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')   # 한국어 문서면 'paraphrase-multilingual-MiniLM-L12-v2'


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
docs = [
    "LLM은 이전 토큰들을 보고 다음 토큰의 확률을 예측하는 모델이다.",
    "임베딩은 텍스트의 의미를 벡터로 바꾸며, 의미가 비슷할수록 벡터가 가깝다.",
    "RAG는 관련 문서를 검색해 프롬프트에 넣고, 그것을 근거로 답을 생성한다.",
    "벡터 DB는 수많은 임베딩 중 질문과 가장 가까운 것을 빠르게 찾는다.",
    "파인튜닝은 모델의 가중치를 직접 학습시켜 행동이나 도메인을 바꾼다.",
    "프롬프트 엔지니어링은 가중치를 바꾸지 않고, 입력 맥락으로 출력을 유도한다.",
]

vecs = model.encode(docs)    # (문서수 × 차원) numpy 배열로 바로 나옴

In [3]:
from langchain_core.documents import Document
docs = [
    "LLM은 이전 토큰들을 보고 다음 토큰의 확률을 예측하는 모델이다.",
    "임베딩은 텍스트의 의미를 벡터로 바꾸며, 의미가 비슷할수록 벡터가 가깝다.",
    "RAG는 관련 문서를 검색해 프롬프트에 넣고, 그것을 근거로 답을 생성한다.",
    "벡터 DB는 수많은 임베딩 중 질문과 가장 가까운 것을 빠르게 찾는다.",
    "파인튜닝은 모델의 가중치를 직접 학습시켜 행동이나 도메인을 바꾼다.",
    "프롬프트 엔지니어링은 가중치를 바꾸지 않고, 입력 맥락으로 출력을 유도한다.",
]
document = [Document(page_content=doc) for doc in docs]
document

[Document(metadata={}, page_content='LLM은 이전 토큰들을 보고 다음 토큰의 확률을 예측하는 모델이다.'),
 Document(metadata={}, page_content='임베딩은 텍스트의 의미를 벡터로 바꾸며, 의미가 비슷할수록 벡터가 가깝다.'),
 Document(metadata={}, page_content='RAG는 관련 문서를 검색해 프롬프트에 넣고, 그것을 근거로 답을 생성한다.'),
 Document(metadata={}, page_content='벡터 DB는 수많은 임베딩 중 질문과 가장 가까운 것을 빠르게 찾는다.'),
 Document(metadata={}, page_content='파인튜닝은 모델의 가중치를 직접 학습시켜 행동이나 도메인을 바꾼다.'),
 Document(metadata={}, page_content='프롬프트 엔지니어링은 가중치를 바꾸지 않고, 입력 맥락으로 출력을 유도한다.')]

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
rcts = RecursiveCharacterTextSplitter(chunk_size= 30,chunk_overlap =5)
chunks = rcts.split_documents(document)


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)

In [7]:
vectorstore.similarity_search("RAG가 뭐야?", k=2)

[Document(id='f2ba7d87-fcd6-4293-8338-259d405b57a8', metadata={}, page_content='확률을 예측하는 모델이다.'),
 Document(id='39c56489-129c-466b-9d4a-c9e7c2ae0cab', metadata={}, page_content='행동이나 도메인을 바꾼다.')]

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(document)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_v2"     # 새 이름 → 완전 별개의 빈 컬렉션
)
vectorstore.similarity_search("RAG가 뭐야?", k=2)

[Document(id='42ba4499-a3cb-48a7-ae6d-909c45872aa6', metadata={}, page_content='RAG는 관련 문서를 검색해 프롬프트에 넣고, 그것을 근거로 답을 생성한다.'),
 Document(id='126b6bb8-d15e-4317-86ab-e22fab7bfc08', metadata={}, page_content='벡터 DB는 수많은 임베딩 중 질문과 가장 가까운 것을 빠르게 찾는다.')]

In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
retriever.invoke("RAG가 뭐야?")

[Document(id='42ba4499-a3cb-48a7-ae6d-909c45872aa6', metadata={}, page_content='RAG는 관련 문서를 검색해 프롬프트에 넣고, 그것을 근거로 답을 생성한다.'),
 Document(id='126b6bb8-d15e-4317-86ab-e22fab7bfc08', metadata={}, page_content='벡터 DB는 수많은 임베딩 중 질문과 가장 가까운 것을 빠르게 찾는다.')]

In [10]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [11]:
llm.invoke("안녕, 한 문장으로 자기소개 해줘")

AIMessage(content='안녕하세요, 저는 사용자와 대화하며 정보를 제공하고 다양한 질문에 답변해 드릴 수 있는 인공지능聊天봇입니다.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 47, 'total_tokens': 81, 'completion_time': 0.161097656, 'completion_tokens_details': None, 'prompt_time': 0.002470995, 'prompt_tokens_details': None, 'queue_time': 0.06328546, 'total_time': 0.163568651}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ebb78-33ca-73a0-8601-3d9558ed2a08-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 34, 'total_tokens': 81})

In [13]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""당신은 AI 전문가이다. 아래 컨텍스트를 근거로만 질문에 답하라.
너가 가진 정보에 없으면 없다고 솔직하게 답해라.
항상 한글과 한국어만을 사용한다.

컨텍스트:
{context}

질문: {question}

답변:""")

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("RAG가 뭐야?")

'RAG는 관련 문서를 검색해 프롬프트에 넣고, 그것을 근거로 답을 생성하는 기술입니다.'

: 